In [ ]:
import os
from brian2 import *
sys.path.append('Neuron and Synapse Models')
from neuronModels import *
from ringAttractorClass import *

sys.path.append('Tools')
from plottingTools import *
from utils import *

set_device('cpp_standalone', build_on_run=False)
mujocoFlag = True
dtMujoco = 0.005  # time step for mujoco simulation

In [ ]:
# Get the absolute path to the project root using the notebook's working directory
project_root = os.getcwd()

# Create absolute paths
spikeMon_file = os.path.join(project_root, 'spikes.txt')
build_dir = os.path.join(project_root, 'standalone_mujoco_build')
tools_dir = os.path.join(project_root, 'Tools')
cpp_path = os.path.join(tools_dir, 'socket_input.cpp')
header_path = os.path.join(tools_dir, 'socket_input.h')

# Print paths for verification
print(f"Project root: {project_root}")
print(f"Build directory: {build_dir}")
print(f"CPP file path: {cpp_path}")
print(f"Header file path: {header_path}")

# Verify file existence
if not os.path.exists(cpp_path):
    raise FileNotFoundError(f"Could not find {cpp_path}")
if not os.path.exists(header_path):
    raise FileNotFoundError(f"Could not find {header_path}")

In [ ]:
@implementation(
    'cpp',
    '''
    #import <cmath>
    double arctan2(double y, double x) {
        return std::atan2(y, x);
        }
    ''')
@check_units(y=1, x=1, result=1)
def arctan2(y, x):
    return np.arctan2(y, x)

In [ ]:
@implementation('cpp', '''
double store_spike(int i, double t)
{
    static std::ofstream spike_file("spikes.txt", std::ios::out | std::ios::app);              // append, don’t truncate
    spike_file << i << ' ' << t << std::endl;       // write spike index and time
    spike_file.flush();                            // <-- critical for visibility
    return 0.;                                     // dummy return
}
''')
@check_units(i=1, t=second, result=1)
def store_spike(i, t):
    raise NotImplementedError('Brian only uses the C++ call store_spike() in standalone mode.')

In [ ]:
@implementation(
    'cpp',
    '// code lives in socket_input.cpp',
    sources=[cpp_path],
    headers=['"{}"'.format(header_path)],
    # Fix the HOST_IP string literal by escaping quotes
    define_macros=[('HOST_IP', '\\"0.0.0.0\\"'), ('PORT_NUM', 5005), ('BUF_BYTES', 1024)],
    include_dirs=[tools_dir])
@check_units(index=1, result=1)
def get_socket_sample(socket_index: int) -> float:
    raise NotImplementedError('Brian only uses the C++ call get_socket_sample() in standalone mode.')

In [ ]:
# Simulation parameters
defaultclock.dt = dtMujoco*second

In [ ]:
# Setting network parameters
num_neurons = 120
tau=10*ms
sigma_noise=0.1*mV
V_rest=-70*mV

In [ ]:
# Creating the equation object
neuron_eq = Equations(LIF_Mujoco, tau=tau, V_rest=V_rest, sigma_noise=sigma_noise)

In [ ]:
# # Connectivity parameters - Mexican hat
# sigma_exc = 0.0875
# sigma_inh = 0.25
# g_exc = 1.0*mV
# g_inh = -0.475*mV

# Testing values
sigma_exc_val= 0.125
sigma_inh_val= 0.25
g_exc= 0.875*mV
g_inh= -0.475*mV

# Connectivity parameters - Cosine
g_cosine = 0.1*mV
w_inh = -0.555*mV

# Create neuron group
Vth=-48*mV
V_reset=-80*mV
refractory_period=5*ms
glob_inh_flag = True
            
ringAttractor = RingAttractor(neuron_eq, 
                        num_neurons, 
                        Vth, V_reset, refractory_period,
                        syn_profile='cosine',
                        autapse=True,
                        glob_inh=glob_inh_flag, w_inh=w_inh,
                        g_cosine=g_cosine,
                        sigma_exc=sigma_exc_val, sigma_inh=sigma_inh_val,
                        g_exc=g_exc, g_inh=g_inh,
                        mujoco=mujocoFlag)  # Set to True for Mujoco simulation)

if not mujocoFlag:
    ringAttractor.ring_pool.I_ext = 0.0 * volt  # Initialize external input to zero
ringAttractor.ring_pool.I_vel = 0.0

ringAttractor.ring_pool.run_regularly('V = clip(V, V_reset, inf*volt)', dt=defaultclock.dt)

In [ ]:
# Setup monitors: spike monitor and state monitor for membrane potential and external input
spikemon = SpikeMonitor(ringAttractor.ring_pool)
statemon = StateMonitor(ringAttractor.ring_pool, 'V', record=True)
inputmon = StateMonitor(ringAttractor.ring_pool, 'I_ext', record=True)  # if you want to check the input

# Set of Brian objects to be added to the network
localObjects = [spikemon, statemon, inputmon] # enforce_lower_bound]

if glob_inh_flag:
    statemon_inh = StateMonitor(ringAttractor.glob_inh_neuron, 'V', record=True)
    spikemon_inh = SpikeMonitor(ringAttractor.glob_inh_neuron)
    localObjects.extend([statemon_inh, spikemon_inh])
    

net = Network(ringAttractor.BrianObjects+localObjects)
net.run(2*second)

In [ ]:
device.build(directory = build_dir, compile=True, run=True, debug=True, clean=True)

In [ ]:
from matplotlib.ticker import FuncFormatter

# Create a figure with 4 subplots arranged in 2 rows and 2 columns#
positions = linspace(0, 2*pi, num_neurons, endpoint=False)
#+---------------------------------------------------------------------------+
#|                           Plotting the Results                            |
#+---------------------------------------------------------------------------+

# Create a figure with 6 subplots arranged in 3 rows and 2 columns
fig = plt.figure(figsize=(15, 15))

# 2. Raster Plot
ax2 = fig.add_subplot(3, 2, 2)
raster_plot(spikemon, ax=ax2)

plt.tight_layout()
plt.show()

In [ ]:
statemon_V = statemon.V[0] / mV  # Convert to mV for plotting
print(f"Statemon V shape: {statemon_V.shape}, first 10 values: {statemon_V[:50]}")